# Data preparation and consolidation

This notebook records the supplied data preparation workflow. It is **not a complete raw-data rebuild**: existing PostgreSQL tables, prepared UNCTAD extracts, mapping workbooks and R-generated intermediate files are required. The upstream SQL and gap-year construction are not supplied by these four notebooks. Read AUDIT.md before execution.

**Status:** publication review candidate; a complete clean-kernel execution has not been verified. Saved outputs and execution counts have been cleared. See README.md for inputs and run order.


## Workspace and configuration


In [ ]:
from pathlib import Path
import os

# Launch Jupyter in the research-notebooks directory (or one of its subdirectories).
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "publication_manifest.json").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the research-notebooks folder.")
WORK = ROOT / "work" / "01_data_preparation"
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print("Working directory:", WORK)
# Required input files belong here; outputs are written here too.
# work/ is excluded from the publication package and from Git by default.


In [ ]:
_database_engine = None

def get_database_engine():
    global _database_engine
    if _database_engine is None:
        from sqlalchemy import create_engine
        database_url = os.environ.get("THESIS_DATABASE_URL")
        if not database_url:
            raise RuntimeError("Set THESIS_DATABASE_URL outside the notebook; do not save credentials in cells.")
        _database_engine = create_engine(database_url)
    return _database_engine


In [ ]:
def write_research_table(frame, name, con, **kwargs):
    if os.environ.get("THESIS_ALLOW_DB_WRITES") != "1":
        raise RuntimeError("ETL replaces database tables. Use an isolated research database and explicitly set THESIS_ALLOW_DB_WRITES=1 to run write steps.")
    return frame.to_sql(name, con, **kwargs)


# cepi

In [ ]:
import pandas as pd

In [ ]:
cepii_tradeprod_data =pd.read_csv("CEPI/TPe_V202401.csv", encoding="utf-8", na_values="")

In [ ]:
cepii_baci_data =pd.read_csv("CEPI/BACI_HS92_V202501/BACI_HS92_Y2023_V202501.csv", encoding="utf-8", na_values="")

In [ ]:
cepii_gravity_data = pd.read_csv("CEPI/Gravity_csv_V202211/Gravity_V202211.csv", encoding="utf-8", na_values="")

In [ ]:

cepii_baci_product_data =pd.read_csv("CEPI/BACI_HS92_V202501/product_codes_HS92_V202501.csv", encoding="utf-8", na_values="")

In [ ]:
cepii_baci_product_data[['description']]
# cepii_baci_product_data

In [ ]:
cepii_gravity_data.columns

In [ ]:
# cepii_gravity_data[
#     (cepii_gravity_data['iso3_o']=='USA')&
#     (cepii_gravity_data['iso3_d']=='FRA')&
#     (cepii_gravity_data['year']==2016)
#     ].head()
import pandas as pd
from sqlalchemy import create_engine
engine = get_database_engine()
write_research_table(cepii_gravity_data, "cepii_gravity_dataset", engine, if_exists="replace", index=False)

In [ ]:
cepii_gravity_data[['country_id_o', 'iso3_o', 'iso3num_o', 'country_exists_o']].drop_duplicates(subset='iso3_o', keep='last')

# FAO Master Data

In [ ]:
import pandas as pd
FAO_balance_sheet = pd.read_csv("FoodBalanceSheets_E_All_Data.csv", encoding="utf-8", na_values="")

In [ ]:
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# Step 1: Load FAO Area Codes CSV
FAO_area_code_file = "FoodBalanceSheets_E_AreaCodes.csv"

df_area_code = pd.read_csv(
    FAO_area_code_file,
    encoding="utf-8",
    na_values=""
)

# Filter only area codes with 4-digit Area.Code and drop duplicates
df_area_code_region = (
    df_area_code[df_area_code[" Area Code"].astype(str).str.len() == 4]
    .drop_duplicates(subset=[" Area Code", " Area"])
)

# ─────────────────────────────────────────────────────────────────────────────
# Step 2: Extract item codes from FAO balance sheet data
# (Assumes FAO_balance_sheet has already been read earlier)
df_item_code = FAO_balance_sheet.drop_duplicates(subset=["Item Code", "Item"])

# ─────────────────────────────────────────────────────────────────────────────
# Step 3: Load masterdata_fao_prod_bom.xlsx
df_masterdata_prod_bom = pd.read_excel("masterdata_fao_prod_bom.xlsx")

# ─────────────────────────────────────────────────────────────────────────────
# Step 4: Merge with item names
FAO_masterdata_prod_bom_rev = (
    df_masterdata_prod_bom
    .merge(df_item_code[["Item Code", "Item"]],
           left_on="product_group_code", right_on="Item Code", how="left")
    .rename(columns={"Item": "product_group_name"})
    .drop(columns="Item Code")
    .merge(df_item_code[["Item Code", "Item"]],
           left_on="category_code", right_on="Item Code", how="left")
    .rename(columns={"Item": "category_name"})
    .drop(columns="Item Code")
)

In [ ]:
FAO_masterdata_prod_bom_rev

# FAO Data

In [ ]:

import pandas as pd
import janitor  # For pivot_longer and pivot_wider
import numpy as np

# Step 0: Load the FAO CSV
FAO_balance_sheet_all_file = "FoodBalanceSheets_E_All_Data.csv"


df_fao = pd.read_csv(
    FAO_balance_sheet_all_file,
    encoding="utf-8",
    na_values=""
)



# Step 1: Filter only the selected elements
selected_elements = [664, 5511, 5611, 5911, 5072]
df_filtered = df_fao[df_fao["Element Code"].isin(selected_elements)]

# Step 2: Keep only relevant columns (years start with Y20), drop F/N suffix
# Select all year columns (e.g., Y2010 to Y2022)
year_cols = df_filtered.columns[df_filtered.columns.str.match(r"^Y20\d{2}$")]
# Define the needed base columns
base_cols = ["Area Code", "Area", "Item Code", "Item", "Element Code", "Unit"]
# Combine into final filtered DataFrame
df_filtered = df_filtered[base_cols + list(year_cols)]

# Step 3: Pivot longer (melt)
df_long = df_filtered.pivot_longer(
    index=["Area Code", "Area", "Item Code", "Item", "Element Code", "Unit"],
    names_to="year",
    values_to="value"
)

# Step 4: Pivot wider – convert Element Codes to columns
df_wide = df_long.pivot_wider(
    index=["Area Code", "Area", "Item Code", "Item", "year"],
    names_from="Element Code",
    names_sep="_",
    values_from=["value", "Unit"]
)

# Step 5: Rename pivoted columns to meaningful names
df_wide = df_wide.rename(columns={
    "value_664": "daily_calorific_food_per_capita",
    "Unit_664": "daily_calorific_food_per_capita_unit",
    "value_5511": "production_qty",
    "Unit_5511": "production_unit",
    "value_5611": "import_qty",
    "Unit_5611": "import_unit",
    "value_5911": "export_qty",
    "Unit_5911": "export_unit",
    "value_5072": "stock_var_qty",
    "Unit_5072": "stock_var_unit"
})
# Step 6: Clean year column
df_wide["year"] = df_wide["year"].str.replace("Y", "").astype(int)

# Step 7: Filter for Area
df_fao_continents = df_wide[df_wide["Area Code"].astype(str).isin(["5100", "5200", "5300", "5400", "5500"])]

# Step 8: Filter for selected item codes (Food category)
selected_items_code = [
    2905,2907,2908,2909,2911,2912,2913,2914,2918,2919,
    2922,2923,2924,2928,2943,2945,2946,2948,2949
]

df_food_category = df_fao_continents[df_fao_continents["Item Code"].isin(selected_items_code)]


# Step 9: Filter for product group code (food group) and join with metadata
# You must first load the FAO_masterdata_prod_bom_rev file as DataFrame:
# Example:
# df_fao_metadata = pd.read_csv("FAO_masterdata_prod_bom_rev.csv")

# Assuming you have already loaded FAO_masterdata_prod_bom_rev
# (make sure its column names match Python conventions)
selected_items_code_foodgroup = FAO_masterdata_prod_bom_rev["product_group_code"]

df_food_group = (
    df_wide[df_wide["Item Code"].isin(selected_items_code_foodgroup)]
    .merge(
        FAO_masterdata_prod_bom_rev[["product_group_code", "category_code", "category_name"]],
        left_on="Item Code",
        right_on="product_group_code",
        how="left"
    )
)

import numpy as np

df_food_group["area_category"] = np.where(
    df_food_group["Area Code"].astype(str).str.len() < 4,
    "country",
    "group"
)

In [ ]:
pd.DataFrame(df_area_code[df_area_code[' Area Code'].astype(str).str.len() < 4][' Area'].unique().tolist(), columns=['Area']).to_excel('fao_country_list.xlsx')

In [ ]:
df_area_code[' Area'].unique()

In [ ]:
import pyreadr

# Load the RData file
result = pyreadr.read_r('df_faodata_supplyfood_per_economicsgroup_foodgroup.RData')
# Inspect available objects
print(result.keys())  # This will show the variable names saved in the RData file
# Access the dataframe (replace with your actual object name)
df_fao_balance_data_food_per_economicsgroup = result['FAO_data_food_per_economicsgroups_foodsgroup']  # Adjust based on the key shown above


In [ ]:
load_fao_balance_data_foodgroup_per_region = pyreadr.read_r('df_faodata_supplyfood_per_continent_foodgroup.RData')
load_fao_balance_data_foodcategory_per_region = pyreadr.read_r('df_faodata_supplyfood_per_continent_foodcategory.RData')

df_load_fao_balance_data_foodcategory_per_region = load_fao_balance_data_foodcategory_per_region['FAO_data_food_per_continent_food_category']
df_load_fao_balance_data_foodgroup_per_region = load_fao_balance_data_foodgroup_per_region['FAO_data_food_per_continent_foodgroup']

In [ ]:
df_load_fao_balance_data_foodcategory_per_region

In [ ]:
# Calculate Import depedency ratio
# df_load_fao_balance_data_foodcategory_per_region['import_depedency_ratio'] = df_load_fao_balance_data_foodcategory_per_region
df_load_fao_balance_data_foodcategory_per_region.columns = [col.strip().replace(" ", "_").lower() for col in df_load_fao_balance_data_foodcategory_per_region.columns]
df_load_fao_balance_data_foodcategory_per_region = (df_load_fao_balance_data_foodcategory_per_region
    .fillna({'production_qty': 0, 'import_qty': 0, 'stock_var_qty': 0})
    .assign(
    import_dependency_ratio = lambda df: df['import_qty'] / (
        df['production_qty'] + df['import_qty'] + df['stock_var_qty'])
    )
)



In [ ]:
# Calculate Import depedency ratio
# df_load_fao_balance_data_foodcategory_per_region['import_depedency_ratio'] = df_load_fao_balance_data_foodcategory_per_region
df_load_fao_balance_data_foodgroup_per_region.columns = [col.strip().replace(" ", "_").lower() for col in df_load_fao_balance_data_foodgroup_per_region.columns]
df_load_fao_balance_data_foodgroup_per_region = (df_load_fao_balance_data_foodgroup_per_region
    .fillna({'production_qty': 0, 'import_qty': 0, 'stock_var_qty': 0})
    .assign(
    import_dependency_ratio = lambda df: df['import_qty'] / (
        df['production_qty'] + df['import_qty'] + df['stock_var_qty']
        )
    )
)
                                                 
                                                 

df_fao_balance_data_food_per_economicsgroup.columns = [col.strip().replace(" ", "_").lower() for col in df_fao_balance_data_food_per_economicsgroup.columns]
df_fao_balance_data_food_per_economicsgroup = (df_fao_balance_data_food_per_economicsgroup
    .fillna({'production_qty': 0, 'import_qty': 0, 'stock_var_qty': 0})
    .assign(
        import_dependency_ratio = lambda df: df['import_qty'] / (df['production_qty'] + df['import_qty'] + df['stock_var_qty'])
    )
)



In [ ]:
df_load_fao_balance_data_foodgroup_per_region

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
engine = get_database_engine()

# Step 5: Save to PostgreSQL
write_research_table(df_fao_balance_data_food_per_economicsgroup, "fao_balance_data_food_per_economicsgroup", engine, if_exists="replace", index=False)
write_research_table(df_load_fao_balance_data_foodcategory_per_region, "fao_balance_data_foodcategory_per_region", engine, if_exists="replace", index=False)
write_research_table(df_load_fao_balance_data_foodgroup_per_region, "fao_balance_data_foodgroup_per_region", engine, if_exists="replace", index=False)

print("✅ Data successfully saved to PostgreSQL.")

**Publication review note.** The table billateral_trade_countries must already exist; its upstream SQL construction is outside this notebook.


In [ ]:
# Pre processing
import pandas as pd
from sqlalchemy import create_engine

# ─────────────────────────────────────────────────────────────────────────────
# STEP 0: Connect to PostgreSQL
engine = get_database_engine()
bilateral_trade_df = pd.read_sql("SELECT * FROM billateral_trade_countries", engine)

In [ ]:
pd.DataFrame(bilateral_trade_df['origin_label'].unique().tolist(), columns=['origin_label']).to_excel('unctad_country_list.xlsx')

In [ ]:
country_fao_unctad_mapping = pd.read_excel("FAO_UNCTAD_Country_Mapping__improved_detection_.xlsx", sheet_name="Sheet 1 - FAO_UNCTAD_Country_Ma")
country_fao_unctad_mapping_fn = country_fao_unctad_mapping[~country_fao_unctad_mapping['FAO_Country'].isin(['China; mainland','Democratic Republic of the Congo'])]

In [ ]:
country_fao_unctad_mapping[country_fao_unctad_mapping.duplicated(keep=False, subset = ['unctad_final_name'])==True]

In [ ]:
df_food_group[df_food_group['Area'].isin(['Congo', 'Democratic Republic of the Congo'])].drop_duplicates(subset=['Area', 'year'], keep='last')

# Master Data Unctad

Push the table to postgre

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# Step 1: Load Excel
df = pd.read_excel("masterdata_country_unctad.xlsx", sheet_name="Sheet1")

# Step 2: Filter rows
filtered_df = df[df["Parent_agg_category"].isin(["region", "sub region"])]
filtered_df["Child_Label_2"] = filtered_df["Child_Label_2"].fillna("MISSING")

# Step 3: Pivot (wide format)
pivoted_df = filtered_df.pivot_table(
    index=["Child_Code", "Child_Label","Child_Label_2","child_isocode3"],
    columns="Parent_agg_category",
    values="Parent_Label",
    aggfunc= "first" , # In case of duplicates
    dropna= True
).reset_index()
##pivoted_df = filtered_df.copy()
# Optional: Clean column names
pivoted_df.columns.name = None
pivoted_df.columns = [str(col).strip().replace(" ", "_").lower() for col in pivoted_df.columns]

# Step 4: Connect to PostgreSQL
engine = get_database_engine()

# Step 5: Save to PostgreSQL
write_research_table(pivoted_df, "master_data_country_unctad", engine, if_exists="replace", index=False)

#print("✅ Data successfully saved to PostgreSQL.")

In [ ]:
pivoted_df

In [ ]:
pivoted_df

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

engine = get_database_engine()
unctad_fao_masterdata_product = pd.read_excel("master data unctad.xlsx")
unctad_fao_masterdata_product.columns = [col.strip().replace(" ", "_").lower() for col in unctad_fao_masterdata_product.columns]
table_name = "unctad_fao_masterdata_product"
write_research_table(unctad_fao_masterdata_product, table_name, engine, if_exists="replace", index=False)
print(f"✅ table '{table_name}' successfully saved to PostgreSQL.")

# Bilateral Trade

## All mode

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ─────────────────────────────────────────────────────────────────────────────
# STEP 0: Connect to PostgreSQL
engine = get_database_engine()

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Load gzipped CSV and clean column names
billat_allmodes_kg = pd.read_csv("merged_UNCTAD_bilateral_allmodes.csv.gz")
billat_allmodes_kg.columns = [col.strip().replace(" ", "_").lower() for col in billat_allmodes_kg.columns]

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Load reference tables from database
master_data_country = pd.read_sql("SELECT * FROM master_data_country_unctad", engine)
product_data = pd.read_sql("SELECT * FROM unctad_fao_masterdata_product", engine)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Merge origin country metadata

# Clean text columns for matching
for col in ["child_label", "child_label_2"]:
    master_data_country[col] = master_data_country[col].str.strip()
billat_allmodes_kg["origin_label"] = billat_allmodes_kg["origin_label"].str.strip()

# Prepare origin reference
origin_ref = master_data_country.rename(columns={
    "child_label": "origin_label",
    "child_label_2": "origin_label_alias",
    "child_code": "origin_code",
    "region": "origin_region",
    "sub_region": "origin_subregion"
})

# First merge on origin_label
billat_allmodes_kg = billat_allmodes_kg.merge(
    origin_ref[['origin_label', 'origin_code', 'origin_region', 'origin_subregion']],
    on='origin_label',
    how='left'
)

# Second merge on alias
alias_origin_ref = origin_ref[["origin_label_alias", "origin_code", "origin_region", "origin_subregion"]].rename(
    columns={"origin_label_alias": "origin_label"}
)
billat_allmodes_kg = billat_allmodes_kg.merge(
    alias_origin_ref, on="origin_label", how="left", suffixes=("", "_alias")
)

# Use .where() to fill missing origin data
for col in ["origin_code", "origin_region", "origin_subregion"]:
    billat_allmodes_kg[col] = billat_allmodes_kg[col].where(
        billat_allmodes_kg[col].notna(),
        billat_allmodes_kg[f"{col}_alias"]
    )

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Merge destination country metadata

# Clean text columns for matching
for col in ["child_label", "child_label_2"]:
    master_data_country[col] = master_data_country[col].str.strip()
billat_allmodes_kg["destination_label"] = billat_allmodes_kg["destination_label"].str.strip()

# Prepare destination reference
destination_ref = master_data_country.rename(columns={
    "child_label": "destination_label",
    "child_label_2": "destination_label_alias",
    "child_code": "destination_code",
    "region": "destination_region",
    "sub_region": "destination_subregion"
})

# First merge on destination_label
billat_allmodes_kg = billat_allmodes_kg.merge(
    destination_ref[['destination_label', 'destination_code', 'destination_region', 'destination_subregion']],
    on='destination_label',
    how='left'
)

# Second merge on alias
alias_destination_ref = destination_ref[["destination_label_alias", "destination_code", "destination_region", "destination_subregion"]].rename(
    columns={"destination_label_alias": "destination_label"}
)
billat_allmodes_kg = billat_allmodes_kg.merge(
    alias_destination_ref, on="destination_label", how="left", suffixes=("", "_alias")
)

# Use .where() to fill missing destination data
for col in ["destination_code", "destination_region", "destination_subregion"]:
    billat_allmodes_kg[col] = billat_allmodes_kg[col].where(
        billat_allmodes_kg[col].notna(),
        billat_allmodes_kg[f"{col}_alias"]
    )

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Merge product metadata
product_ref = product_data.rename(columns={
    "parent_label": "product_parent_label"
})
billat_allmodes_kg = billat_allmodes_kg.merge(
    product_ref[["product_label", "product_code", "product_parent_label"]],
    on="product_label",
    how="left"
)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Final column cleanup
billat_allmodes_kg.columns = [col.strip().replace(" ", "_").lower() for col in billat_allmodes_kg.columns]
billat_allmodes_kg.drop(columns=[col for col in billat_allmodes_kg.columns if col.endswith("_alias")], inplace=True)

## Sea Mode

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
# Step 1: Read gzipped CSV
billat_seamodes_kg = pd.read_csv("merged_UNCTAD_bilateral_seamodes.csv.gz")
billat_seamodes_kg.columns = [col.strip().replace(" ", "_").lower() for col in billat_seamodes_kg.columns]


In [ ]:
merge_billateral_trade = billat_allmodes_kg.rename(columns = {
    "kilograms_value" : "kilograms_value_allmodes",
    "kilograms_missingvalue" : "kilograms_missingvalue_allmodes"
})
billat_seamodes_kg = billat_seamodes_kg.rename(columns = {
    "kilograms_value" : "kilograms_value_seamodes",
    "kilograms_missingvalue" : "kilograms_missingvalue_seamodes"
})

merge_billateral_trade = merge_billateral_trade.merge(
    billat_seamodes_kg,
    on=["origin_label", "product_label","destination_label","year"],
    how="left")

In [ ]:
merge_billateral_trade['seatrade_cnt_vol'] = merge_billateral_trade['kilograms_value_seamodes']/merge_billateral_trade['kilograms_value_allmodes']

In [ ]:
print(merge_billateral_trade.columns)

In [ ]:
merge_billateral_trade[(merge_billateral_trade['origin_label']=='United States of America')&(merge_billateral_trade['origin_label'].isna())]

In [ ]:
# Step 6: Save to PostgreSQL
write_research_table(merge_billateral_trade, "billat_allmodes_kg", engine, if_exists="replace", index=False)

print("✅ Data processed and saved to PostgreSQL as 'billat_allmodes_kg'")

In [ ]:
# Load Data
import pandas as pd
from sqlalchemy import create_engine

engine = get_database_engine()
billateral_trade_indvidual = pd.read_sql("SELECT * FROM billateral_trade_countries", engine)

In [ ]:
billateral_trade_indvidual.columns

In [ ]:
billateral_trade_indvidual[(billateral_trade_indvidual['origin_label']=="United States of America")&(billateral_trade_indvidual['destination_label']=="China")]

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
billat_allmodes_kg = pd.read_csv("merged_UNCTAD_bilateral_allmodes.csv.gz")

In [ ]:
billat_allmodes_kg[billat_allmodes_kg['Origin_Label']=='Republic of Korea']

### Transport intensity

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

engine = get_database_engine()
billat_allmodes_kg = pd.read_sql("SELECT * FROM billat_allmodes_kg", engine)
product_masterdata = pd.read_sql("SELECT * FROM unctad_fao_masterdata_product", engine)

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ─────────────────────────────────────────────────────────────────────────────
# STEP 0: Connect to PostgreSQL
engine = get_database_engine()

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Load gzipped CSV and clean column names
billat_seamodes_inst = pd.read_csv("merged_UNCTAD_bilateral_seamodes_transpints.csv.gz")
billat_seamodes_inst.columns = [col.strip().replace(" ", "_").lower() for col in billat_seamodes_inst.columns]

In [ ]:
product_masterdata["missing_in_inst"] = ~product_masterdata["product_label"].isin(
    billat_seamodes_inst["product_label"]
)
    

In [ ]:
billat_seamodes_inst

In [ ]:
write_research_table(billat_seamodes_inst, "billat_seamodes_inst", engine, if_exists="replace", index=False)

print("✅ Data processed and saved to PostgreSQL as 'billat_seamodes_inst'")

## Add additional indicators/data from UNCTAD Transport database

In [ ]:
billat_seamods_transfric = pd.read_csv("merged_UNCTAD_bilateral_seamodes_by_code.csv.gz")

**Publication review note.** This section reads an existing billateral_trade_combined_data table before updating it. The later FAO/economic write replaces that table again; verify the final schema before using downstream notebooks.


In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
billat_seamods_transfric = pd.read_csv("merged_UNCTAD_bilateral_seamodes_by_code.csv.gz")
engine = get_database_engine()
billat_trade_combined_data =  pd.read_sql("SELECT * FROM billateral_trade_combined_data", engine)

In [ ]:
#billat_seamods_transfric[billat_seamods_transfric['Origin_Code']=='380']
billat_seamods_transfric['Origin_Label'].unique()

In [ ]:
billat_seamods_transfric.columns = [col.strip().replace(" ", "_").lower() for col in billat_seamods_transfric.columns]
billat_seamods_transfric_1 = billat_seamods_transfric.rename(columns = {
    "transport_expenditure_us_value" : "transport_expenditure_us_value_seamodes",
    "transport_work_in_tonkm_value" : "transport_work_in_tonkm_value_seamodes"
})
billat_trade_combined_data = billat_trade_combined_data.drop(columns=['transport_expenditure_us_value_seamodes','transport_work_in_tonkm_value_seamodes'])
billat_trade_combined_data_mergeg_transfric_1 = billat_trade_combined_data.merge(
    billat_seamods_transfric_1[['origin_label','destination_label','product_label','year','transport_expenditure_us_value_seamodes', 'transport_work_in_tonkm_value_seamodes']],
    on=["origin_label", "product_label","destination_label","year"],
    how="left")

In [ ]:
billat_trade_combined_data_mergeg_transfric_1.columns

In [ ]:
#billat_trade_combined_data_mergeg_transfric_1.to_sql("billateral_trade_combined_data", engine, if_exists="replace", index=False)

In [ ]:
# LSBCI

In [ ]:
df_lbsci = pd.read_csv("data_LSBCI_raw__unctad_20230712054016.csv")

In [ ]:
df_lbsci

In [ ]:
# Step 1: Clean column names (lowercase and replace spaces with underscores)
df_lbsci.columns = df_lbsci.columns.str.strip().str.lower().str.replace(" ", "_")
df_lbsci = df_lbsci.dropna(subset=["index"])
# Step 3: Extract year from 'quarter' column
df_lbsci["year"] = df_lbsci["quarter"].str[:4].astype(int)
df_lbsci_agg = (
    df_lbsci.groupby(["year", "economy_label", "partner_label"], as_index=False)
    .agg(mean_index=("index", "mean"))
    .sort_values(by=["year", "economy_label", "partner_label"])
)

In [ ]:
df_lbsci_agg

In [ ]:
df_lbsci_agg = df_lbsci_agg.rename(columns = {
    "economy_label" : "origin_label",
    "partner_label" : "destination_label",
    "mean_index" : "lsbci"
})

billat_trade_combined_data_mergeg_transfric_2 = billat_trade_combined_data_mergeg_transfric_1.merge(
    df_lbsci_agg[['origin_label','destination_label','year','lsbci']],
    on=["origin_label","destination_label","year"],
    how="left")

In [ ]:
billat_trade_combined_data_mergeg_transfric_2.head()

In [ ]:
write_research_table(billat_trade_combined_data_mergeg_transfric_2, "billateral_trade_combined_data", engine, if_exists="replace", index=False)

# Join Data Billateral Trade x FAO

In [ ]:
df_fao_food_group = (
    df_food_group
    .merge(
        country_fao_unctad_mapping_fn[['unctad_final_name','FAO_Country']],
        left_on="Area",
        right_on="FAO_Country",
        how="left"
    )
    .drop(columns=['FAO_Country'])
)

In [ ]:
df_fao_food_group = (
    df_fao_food_group
    .merge(
        product_data[~product_data['fao_label'].isna()][['product_code','product_label','fao_label']],
        left_on="Item",
        right_on="fao_label",
        how="left"
    )
    .drop(columns=['fao_label'])
)



In [ ]:
df_fao_food_group[['unctad_final_name','year','product_label','daily_calorific_food_per_capita','production_qty','daily_calorific_food_per_capita_unit','production_unit']]

In [ ]:
df_fao_food_group.columns

In [ ]:
bilateral_trade_df_fao = (
    bilateral_trade_df
    .merge(
        df_fao_food_group[['unctad_final_name','year','product_label','daily_calorific_food_per_capita','production_qty','daily_calorific_food_per_capita_unit','production_unit']],
        left_on=["origin_label",'year','product_label'],
        right_on=["unctad_final_name",'year','product_label'],
        how="left"
    )
    .rename(columns={
    "daily_calorific_food_per_capita": "origin_daily_calorific_food_per_capita",
    "production_qty": "origin_production_qty"
    }) 
)


In [ ]:
bilateral_trade_df_fao.columns

In [ ]:
bilateral_trade_df_fao = (
    bilateral_trade_df_fao
    .drop(columns=['unctad_final_name'])
    .merge(
        df_fao_food_group[['unctad_final_name','year','product_label','daily_calorific_food_per_capita','production_qty']],
        left_on=["destination_label",'year','product_label'],
        right_on=["unctad_final_name",'year','product_label'],
        how="left"
    )
    .rename(columns={
    "daily_calorific_food_per_capita": "destination_daily_calorific_food_per_capita",
    "production_qty": "destination_production_qty"
    }) 
)


In [ ]:
product_data = pd.read_sql("SELECT * FROM unctad_fao_masterdata_product", engine)


In [ ]:
bilateral_trade_df

In [ ]:
bilateral_trade_df.dtypes

# Economic Data

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_worldbank = pd.read_csv('WorldBank_20052024_GDP_Population_Data.csv')

In [ ]:
df_worldbank_filtered = df_worldbank[~df_worldbank['Time'].isin(['Data from database: World Development Indicators','Last Updated: 07/01/2025'])]
df_worldbank_cleaned = df_worldbank_filtered.dropna(subset=['Time']).astype({'Time': 'int32'})

In [ ]:
df_worldbank_cleaned['GDP (current US$) [NY.GDP.MKTP.CD]'] = pd.to_numeric(
    df_worldbank_cleaned['GDP (current US$) [NY.GDP.MKTP.CD]'], errors='coerce'
).fillna(0)
df_worldbank_cleaned['Population, total [SP.POP.TOTL]'] = pd.to_numeric(
    df_worldbank_cleaned['Population, total [SP.POP.TOTL]'], errors='coerce'
).fillna(0)

In [ ]:
df_worldbank_cleaned = df_worldbank_cleaned.rename(columns={'GDP (current US$) [NY.GDP.MKTP.CD]': 'gdp_usd_current','Population, total [SP.POP.TOTL]': 'population_total'})

In [ ]:
df_worldbank_cleaned

**Publication review note.** Removed an obsolete cast referring to column names already renamed above. GDP remains numeric; it is not coerced to 32-bit integers.


In [ ]:
#pd.DataFrame(df_worldbank_cleaned['Country Name'].unique(),columns=['wb_Country Name']).to_excel('worldbank_data.xlsx')

In [ ]:
worldbank_to_unctad_country_mapping = pd.read_excel('worldbank_to_unctad_country_mapping.xlsx',sheet_name="mapping")
worldbank_to_unctad_country_mapping_fn = worldbank_to_unctad_country_mapping.copy()

In [ ]:
worldbank_to_unctad_country_mapping_fn['unctad_final_name'] = np.where(worldbank_to_unctad_country_mapping_fn['Score'] <=90,np.nan,worldbank_to_unctad_country_mapping_fn['UNCTAD_Match'])

In [ ]:
worldbank_to_unctad_country_mapping_fn[(worldbank_to_unctad_country_mapping_fn.duplicated(keep=False, subset = ['unctad_final_name'])==True)&(~worldbank_to_unctad_country_mapping_fn['unctad_final_name'].isna())]

In [ ]:
worldbank_to_unctad_country_mapping_fn.loc[206, 'unctad_final_name'] = 'United States of America'
worldbank_to_unctad_country_mapping_fn.loc[117, 'unctad_final_name'] = 'China, Macao SAR'
worldbank_to_unctad_country_mapping_fn.loc[158, 'unctad_final_name'] = np.nan
worldbank_to_unctad_country_mapping_fn.loc[212, 'unctad_final_name'] = np.nan

In [ ]:
df_worldbank_cleaned = (
    df_worldbank_cleaned
    .merge(
        worldbank_to_unctad_country_mapping_fn[['WB_Country','unctad_final_name']],
        left_on="Country Name",
        right_on="WB_Country",
        how="left"
    )
    .drop(columns=['WB_Country'])
)


In [ ]:
df_worldbank_cleaned

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
engine = get_database_engine()
# Step 4: Connect to PostgreSQL
engine = get_database_engine()

# Step 5: Save to PostgreSQL
write_research_table(df_worldbank_cleaned, "data_worldbank_cleaned", engine, if_exists="replace", index=False)

print("✅ Data successfully saved to PostgreSQL.")

# Join data billateral trade x Economic data

In [ ]:
bilateral_trade_df_fao_econ = (
    bilateral_trade_df_fao.drop(columns=['unctad_final_name'])
    .merge(
        df_worldbank_cleaned[['Time','unctad_final_name','gdp_usd_current','population_total']],
        left_on=["origin_label",'year'],
        right_on=["unctad_final_name",'Time'],
        how="left"
    )
    .drop(columns=['Time','unctad_final_name'])
    .rename(columns ={
    "gdp_usd_current": "origin_gdp_usd_current",
    "population_total": "origin_population_total"
    })
)
bilateral_trade_df_fao_econ = (
    bilateral_trade_df_fao_econ
    .merge(
        df_worldbank_cleaned[['Time','unctad_final_name','gdp_usd_current','population_total']],
        left_on=["destination_label",'year'],
        right_on=["unctad_final_name",'Time'],
        how="left"
    )
    .drop(columns=['Time','unctad_final_name'])
    .rename(columns ={
    "gdp_usd_current": "destination_gdp_usd_current",
    "population_total": "destination_population_total"
    })
)



In [ ]:
bilateral_trade_df_fao_econ.columns

In [ ]:
# Step 4: Connect to PostgreSQL
engine = get_database_engine()

# Step 5: Save to PostgreSQL
write_research_table(bilateral_trade_df_fao_econ, "billateral_trade_combined_data", engine, if_exists="replace", index=False)

print("✅ Data successfully saved to PostgreSQL.")

# FBX and BDI data

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
engine = get_database_engine()

In [ ]:
## Source: https://www.investing.com/indices/baltic-dry-historical-data
bdimonthly_file = "Baltic Dry Index Historical Data_2016-2024.csv"
df_bdimonthly_raw = pd.read_csv(
    bdimonthly_file,
    encoding="utf-8",
    na_values=""
)

In [ ]:
df_bdimonthly_raw['Date'] = pd.to_datetime(df_bdimonthly_raw['Date'], format='%m/%d/%Y', errors='coerce')
df_bdimonthly_raw['month'] = df_bdimonthly_raw['Date'].dt.month
df_bdimonthly_raw['year'] = df_bdimonthly_raw['Date'].dt.year
df_bdimonthly_raw = df_bdimonthly_raw[['Date', 'year', 'month', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']]
numeric_cols = ['Price', 'Open', 'High', 'Low']
for col in numeric_cols:
    df_bdimonthly_raw[col] = (
        df_bdimonthly_raw[col]
        .astype(str)          # ensure string
        .str.replace(',', '', regex=False)
        .astype(float)
    )


In [ ]:
df_bdi_yearly = df_bdimonthly_raw.groupby(["year", "month"], as_index=False).agg(
    {
        "Price": "mean"
    }
)

In [ ]:
df_bdi_yearly = df_bdi_yearly.groupby(["year"], as_index=False).agg(
    {
        "Price": "mean"
    }
)

In [ ]:
write_research_table(df_bdi_yearly, "balticdryindex_data_yearly", engine, if_exists="replace", index=False)

In [ ]:
df_bdi_yearly

**Publication review note.** The following PortWatch material is retained as supplementary exploration. Its role in the final model is not established here.


# PortWatch

## From Port Activity

https://portwatch.imf.org/datasets/959214444157458aad969389b3ebe1a0_0/explore

In [ ]:
pwdailyportdata_file = "PW_Daily_Port_Activity_Data_and_Trade_Estimates.csv"

In [ ]:
#The rest of data cleansing is done in the sql

In [ ]:
import pandas as pd


In [ ]:
pwdailyportdata_file = "PW_Daily_Port_Activity_Data_and_Trade_Estimates.csv"
df_pwdailyportdata_raw = pd.read_csv(
    pwdailyportdata_file,
    encoding="utf-8",
    na_values=""
)

In [ ]:
df_pw_portactivitydata_agg_day = df_pwdailyportdata_raw.groupby(["year","month","day", "portname","portid", "country","ISO3"], as_index=False
                                                       ).agg({
        "import_container": "sum",
        "import_dry_bulk": "sum",
        "import_general_cargo": "sum",
        "import_roro": "sum",
        "import_tanker": "sum",
        "export_container": "sum",
        'export_dry_bulk': "sum", 
        'export_general_cargo': "sum",
        'export_roro': "sum",
        'export_tanker': "sum",        
        'portcalls_container': "sum",
        'portcalls_dry_bulk': "sum",
        'portcalls_general_cargo': "sum",
        'portcalls_roro': "sum", 
        'portcalls_tanker': "sum"
    }).sort_values(by=["year","month","day"])
df_pw_portactivitydata_agg_day["date"] = pd.to_datetime(df_pw_portactivitydata_agg_day[["year", "month", "day"]])
df_pw_portactivitydata_agg_day["import_unit"] = 'metric ton'
df_pw_portactivitydata_agg_day["export_unit"] = 'metric ton'

In [ ]:
df_pw_portactivitydata_agg_year = df_pw_portactivitydata_agg_day.groupby(["year", "portname","portid", "country"], as_index=False
                                                       ).agg({
        "import_container": "sum",
        "import_dry_bulk": "sum",
        "import_general_cargo": "sum",
        "import_roro": "sum",
        "import_tanker": "sum",
        "export_container": "sum",
        'export_dry_bulk': "sum", 
        'export_general_cargo': "sum",
        'export_roro': "sum",
        'export_tanker': "sum"
    }).sort_values(by=["year"])

In [ ]:
df_pw_portactivitydata_agg_year['year'].max()

In [ ]:
df_pw_portactivitydata_agg_day[["country","ISO3"]].drop_duplicates(subset=["country","ISO3"]).to_excel('imfpw_country_list.xlsx')

In [ ]:
df_pw_portactivitydata_agg_day[df_pw_portactivitydata_agg_day["country"]=='Indonesia']["portname"].unique()

In [ ]:
write_research_table(df_pw_portactivitydata_agg_day, "pw_portactivitydata_daily", engine, if_exists="replace", index=False)

## Port Supply Chain

https://www.nature.com/articles/s41467-022-32070-0#data-availability

### Check Data

In [ ]:
cepii_baci_data =pd.read_csv("CEPI/BACI_HS92_V202501/BACI_HS92_Y2023_V202501.csv", encoding="utf-8", na_values="")